<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/serialize%20kv%20store.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import random
import string

def generate_random_alphanumeric_string(length):
    characters = string.ascii_letters + string.digits
    return ''.join(random.choice(characters) for i in range(length))

def generate_random_full_string(length):
    characters = string.ascii_letters + string.digits + string.punctuation
    return ''.join(random.choice(characters) for i in range(length))

random_dict = {}
for _ in range(10):
    key = generate_random_alphanumeric_string(6)
    value = generate_random_full_string(10)
    random_dict[key] = value



In [23]:
from struct import pack_into, unpack_from

class TLVencoder:
    def __init__(self) -> None:
        pass

    def _encode(self, data, kv):
        hdr = bytearray(5)
        body = bytearray(data, "utf-8")
        pack_into("<BL", hdr, 0, 1 if kv else 0, len(body))
        return hdr + body

    def encode(self, kv_store):
        for k, v in kv_store.items():
            yield self._encode(k, True)
            yield self._encode(v, False)

    def _decode(self, data, offset):
        t, len = unpack_from("<BL", data, offset)
        m = memoryview(data)
        s = bytearray(m[offset + 5: offset + 5 + len]).decode("utf-8")
        return s, t, offset + 5 + len

    def decode(self, data):
        offset = 0
        while offset < len(data):
            k, _, offset = self._decode(data, offset)
            v, _, offset = self._decode(data, offset)
            yield k, v


In [25]:
print(random_dict)
T = TLVencoder()
blob = bytearray()
for b in T.encode(random_dict):
    blob += b
new_dict = {k:v for k, v in T.decode(blob)}
print(new_dict)

{'3oEFUO': 'C72F!PodH:', 'S6Sayo': 'RB$032(^`x', 'bsmN8d': 'K_4,C*mC<i', 'Uwn0oq': 'l#"#%.2~aE', '0Z2wZI': 'j1EXWxl@^v', 'mYRjPx': 'HpQVH;UH<;', 'JTwn3l': 'dX|o~&6fJ3', 'AuLbqw': '!oS#0{%D-D', 'No83Px': 'Bn*u"JVWyX', 'niQw4O': 'pTWXSVoA+!'}
{'3oEFUO': 'C72F!PodH:', 'S6Sayo': 'RB$032(^`x', 'bsmN8d': 'K_4,C*mC<i', 'Uwn0oq': 'l#"#%.2~aE', '0Z2wZI': 'j1EXWxl@^v', 'mYRjPx': 'HpQVH;UH<;', 'JTwn3l': 'dX|o~&6fJ3', 'AuLbqw': '!oS#0{%D-D', 'No83Px': 'Bn*u"JVWyX', 'niQw4O': 'pTWXSVoA+!'}
